# Arizona Restaurant Analysis System

## Problem Statement
The analytical problem is that diners in Arizona suffer decision fatigue due to numerous restaurants, messy reviews, unknown wait times, and inconsistent quality.

## Target Users
The intended users are people preparing to dine out who need support in choosing restaurants and planning dining time.

## Cell 1: Imports and Environment Setup
### Overview
This cell initializes the working environment for the data preprocessing pipeline.

### Key Functions
1.  **Library Imports**: Loads core dependencies (`json`, `pandas`, `pickle`, `os`) for data handling and I/O operations.
2.  **Directory Setup**: Creates a dedicated folder to store all processed output files.

In [ ]:
# Cell 1: Imports and Setup
import json
import pandas as pd
import pickle
import os

SAVE_DIR = os.getenv("RESTAURANT_DATA_DIR", "./my_restaurant_data")
os.makedirs(SAVE_DIR, exist_ok=True)

## Cell 2: Restaurant Data Preprocessing & Cuisine Standardization
### Overview
This cell handles the core data cleaning and filtering of the Yelp academic dataset.

### Key Functions
1.  **Cuisine Mapping**: Standardizes messy category tags into 10 main cuisine types (e.g., Chinese, Italian, Japanese).
2.  **Data Filtering**: Retains only businesses that meet specific quality criteria.

### Filter Logic
Keep restaurants where:
*   State = **AZ (Arizona)**
*   Category includes *Restaurant* or *Food*
*   Stars ≥ **3.0**
*   Currently **open**
*   At least **10 reviews**

### Output
Generates a clean DataFrame (`df_restaurants`) with a standardized `cuisine` column, saved as both CSV and compressed Parquet files for downstream analysis.

In [ ]:
# Cell 2: Process and Save Restaurants with Cuisine Mapping and State Filter

print("Processing restaurants...")
print("Filters: Restaurant/Food in categories, stars>=3.0, is_open=1, review_count>=10, state='AZ'")

CUISINE_MAPPING = {
    'Chinese': ['Chinese', 'Dim Sum', 'Szechuan', 'Cantonese', 'Hot Pot'],
    'Italian': ['Italian', 'Pizza', 'Pasta', 'Sicilian', 'Roman'],
    'Japanese': ['Japanese', 'Sushi', 'Ramen', 'Izakaya', 'Teppanyaki', 'Tempura'],
    'Mexican': ['Mexican', 'Tacos', 'Burritos', 'Tex-Mex'],
    'American': ['American', 'Burgers', 'BBQ', 'Steakhouses', 'Diners'],
    'French': ['French', 'Bistro', 'Creperies'],
    'Korean': ['Korean', 'Korean BBQ'],
    'Thai': ['Thai'],
    'Indian': ['Indian', 'Curry'],
    'Vietnamese': ['Vietnamese', 'Pho'],
    'Mediterranean': ['Mediterranean', 'Greek', 'Middle Eastern', 'Lebanese'],
    'Other': []
}

def extract_cuisine(categories_str):
    if not categories_str:
        return 'Other'
    
    categories_list = [c.strip() for c in categories_str.split(',')]
    
    for cuisine, keywords in CUISINE_MAPPING.items():
        if cuisine == 'Other':
            continue
        for keyword in keywords:
            if any(keyword.lower() in cat.lower() for cat in categories_list):
                return cuisine
    
    return 'Other'

business_list = []
with open("/Users/zhangyiting/Downloads/archive/yelp_academic_dataset_business.json", 'r') as f:
    for line in f:
        biz = json.loads(line.strip())
        
        # Filter: only Arizona state
        state = biz.get('state', '')
        if state != 'AZ':
            continue
        
        categories = biz.get('categories', '') or ''
        if 'Restaurant' not in categories and 'Food' not in categories:
            continue
        
        stars = biz.get('stars', 0)
        if stars < 3.0:
            continue
        
        is_open = biz.get('is_open', 0)
        if is_open != 1:
            continue
        
        review_count = biz.get('review_count', 0)
        if review_count < 10:
            continue
        
        cuisine = extract_cuisine(categories)
        
        business_list.append({
            'business_id': str(biz['business_id']),
            'name': biz['name'],
            'address': biz.get('address', ''),
            'city': biz.get('city', ''),
            'state': state,
            'postal_code': biz.get('postal_code', ''),
            'latitude': biz.get('latitude'),
            'longitude': biz.get('longitude'),
            'stars': stars,
            'review_count': review_count,
            'is_open': is_open,
            'categories': categories,
            'cuisine': cuisine
        })

df_restaurants = pd.DataFrame(business_list)
df_restaurants = df_restaurants[df_restaurants['cuisine'] != 'Other']

df_restaurants.to_csv(f"{SAVE_DIR}/restaurants.csv", index=False)
df_restaurants.to_parquet(f"{SAVE_DIR}/restaurants.parquet", compression='gzip')

print(f"Saved: {len(df_restaurants)} restaurants (Arizona only)")
print(f"Cuisine distribution:")
print(df_restaurants['cuisine'].value_counts())

restaurant_ids = set(df_restaurants['business_id'].tolist())


## Cell 3: Review Data Processing & Saving
### Overview
This cell processes and filters the Yelp review dataset, linking reviews to the preprocessed restaurant list.

### Key Functions
1.  **Data Filtering**: Retains only reviews for valid Arizona restaurants, with a max of 30 reviews per restaurant.
2.  **Performance Optimization**: Caps total reviews at 500,000 to speed up processing.
3.  **Data Structuring**: Enriches reviews with restaurant metadata (name, cuisine, etc.).

### Output
Saves processed reviews as a CSV file and a Pickle dictionary for efficient access in the Streamlit app.

In [ ]:
# Cell 3: Process and Save Reviews 
print("Processing reviews...")
print("Filters: in restaurant list, max 30 per restaurant")
print("ADDED: Stop at 500,000 reviews to speed up")  

df_restaurants = pd.read_csv(f"{SAVE_DIR}/restaurants.csv")
restaurant_ids = set(df_restaurants['business_id'].astype(str).tolist())

reviews_list = []
reviews_dict = {}
count = 0
max_reviews = 500000 

with open("/Users/zhangyiting/Downloads/archive/yelp_academic_dataset_review.json", 'r') as f:
    for line in f:
        count += 1
        if count % 100000 == 0:
            print(f"  Processed {count:,}...")
        
        if len(reviews_list) >= max_reviews:
            print(f"  Reached limit: {max_reviews} reviews, stopping...")
            break
        
        review = json.loads(line.strip())
        biz_id = str(review['business_id'])
        
        if biz_id not in restaurant_ids:
            continue
        
        if biz_id not in reviews_dict:
            reviews_dict[biz_id] = []
        
        if len(reviews_dict[biz_id]) >= 30:
            continue
        
        restaurant = df_restaurants[df_restaurants['business_id'] == biz_id].iloc[0]
        
        review_data = {
            'business_id': biz_id,
            'name': restaurant['name'],
            'categories': restaurant['categories'],
            'stars': review['stars'],
            'comment': review.get('text', '')[:300],  
            'date': review['date']
        }
        
        reviews_dict[biz_id].append(review_data)
        reviews_list.append(review_data)

df_reviews = pd.DataFrame(reviews_list)
df_reviews.to_csv(f"{SAVE_DIR}/reviews.csv", index=False)

with open(f"{SAVE_DIR}/reviews.pkl", 'wb') as f:
    pickle.dump(reviews_dict, f)

print(f"✓ Saved: {len(reviews_list)} reviews")
print(f"  - CSV: reviews.csv")
print(f"  - Pickle: reviews.pkl")


## Cell 4: Check-in Data Processing & Saving
### Overview
This cell processes Yelp check-in data to analyze restaurant foot traffic patterns.

### Key Functions
1.  **Data Filtering**: Retains only check-ins for valid Arizona restaurants.
2.  **Hourly Aggregation**: Calculates check-in counts for each hour of the day.
3.  **Peak Hour Calculation**: Identifies the busiest hour for each restaurant.

### Output
Saves processed check-in data as a CSV file and a Pickle dictionary, including hourly traffic and peak hour metrics for the Streamlit dashboard.

In [ ]:
# Cell 4: Process and Save Checkins
print("Processing checkins...")

df_restaurants = pd.read_csv(f"{SAVE_DIR}/restaurants.csv")
restaurant_ids = set(df_restaurants['business_id'].astype(str).tolist())

checkin_list = []
checkin_dict = {}
count = 0

with open("/Users/zhangyiting/Downloads/archive/yelp_academic_dataset_checkin.json", 'r') as f:
    for line in f:
        count += 1
        if count % 50000 == 0:
            print(f"  Processed {count:,}...")
        
        checkin = json.loads(line.strip())
        biz_id = str(checkin['business_id'])
        
        if biz_id not in restaurant_ids:
            continue
        
        dates = checkin.get('date', '').split(',')
        
        hourly = defaultdict(int)
        for d in dates[:100]:  
            try:
                hour = int(d.strip().split()[1].split(':')[0])
                hourly[f'h{hour}'] += 1
            except:
                continue
        
        peak = max(hourly, key=hourly.get) if hourly else None
        
        restaurant = df_restaurants[df_restaurants['business_id'] == biz_id].iloc[0]
        
        checkin_data = {
            'business_id': biz_id,
            'name': restaurant['name'],
            'peak_hour': peak,
            'total_checkin': len(dates)
        }
        for h in range(24):
            checkin_data[f'h{h}'] = hourly.get(f'h{h}', 0)
        
        checkin_dict[biz_id] = {
            'name': restaurant['name'],
            'peak_hour': peak,
            'total_checkin': len(dates),
            'hourly': dict(hourly)
        }
        
        checkin_list.append(checkin_data)

df_checkin = pd.DataFrame(checkin_list)
df_checkin.to_csv(f"{SAVE_DIR}/checkins.csv", index=False)

with open(f"{SAVE_DIR}/checkin.pkl", 'wb') as f:
    pickle.dump(checkin_dict, f)

print(f"✓ Saved: {len(checkin_list)} checkins")
print(f"  - CSV: checkins.csv")
print(f"  - Pickle: checkin.pkl")
